# Month 2, Step 1: Field Completeness Audit

Checks completeness of the six fields named in the brief, on the sample fetched by `01_fetch_sample.py`:

- `skills`
- `occupation_uris`
- `content` / `description`
- `location` (with NUTS codes where available)
- `upload_date` / `start_date`
- source identifier

Field names below are confirmed against the live `ProfileSchema` / `JobSchema` definitions in the API's Swagger docs (2026-09-29), not guessed. Three things worth noting up front:

1. **`occupation_uris` only exists on profiles.** The job/posting equivalent is called `occupations` (same idea, different name/shape).
2. **NUTS codes exist only for jobs** (`nuts1`, `nuts2`, `nuts3`, flat fields), **not for profiles**. Profiles only have free-text `location`, `city`, `country`, `region`. This means the location/NUTS completeness check will show 0% for profiles by construction, not by bug -- this is itself a finding worth reporting (see brief's "Greece panel reliability" concern).
3. **Profiles have both `content` and `description` fields** where the brief's wording ("content/description") suggested one -- both are checked separately below.

In [1]:
import json
import pandas as pd

from pathlib import Path

pd.set_option('display.max_columns', None)

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

profiles = load_jsonl('data/raw/profiles_sample.jsonl')
postings = load_jsonl('data/raw/postings_sample.jsonl')

print('Profiles:', profiles.shape)
print('Postings:', postings.shape)

# Run this to see the ACTUAL field names before trusting FIELD_MAP below
print('\nProfile columns:', profiles.columns.tolist())
print('\nPosting columns:', postings.columns.tolist())

Profiles: (7222, 28)
Postings: (6723, 17)

Profile columns: ['skills', 'sectors', 'id', 'full_name', 'location', 'content', 'occupation', 'occupation_uris', 'city', 'company', 'country', 'degree', 'description', 'ethnicity_predicted', 'highest_degree', 'region', 'sex_predicted', 'user_country', 'user_location', 'startdate', 'university_name', 'university_raw', 'university_country', 'university_location', 'ultimate_parent_school_name', 'url', 'source', 'source_id']

Posting columns: ['organization', 'skills', 'occupations', 'sectors', 'id', 'title', 'description', 'experience_level', 'type', 'location', 'location_code', 'nuts1', 'nuts2', 'nuts3', 'upload_date', 'source', 'source_id']


In [2]:
# Confirmed against live ProfileSchema / JobSchema (2026-09-29).

PROFILE_FIELDS = {
    'skills': 'skills',
    'occupation_uris': 'occupation_uris',   # object, profiles only
    'content': 'content',
    'description': 'description',           # profiles have BOTH -- check separately
    'location': 'location',                 # free text; no NUTS fields exist on profiles
    'date': 'startdate',                    # note: no underscore, per the live schema
    'source': 'source',                     # real field, confirmed -- no longer a filter hack
}

POSTING_FIELDS = {
    'skills': 'skills',
    'occupations': 'occupations',           # array<string>; job equivalent of occupation_uris
    'description': 'description',
    'location': 'location',
    'nuts1': 'nuts1',
    'nuts2': 'nuts2',
    'nuts3': 'nuts3',
    'date': 'upload_date',
    'source': 'source',
}

In [3]:
def is_non_empty(value):
    """Treat None, NaN, empty string, and empty list/dict as 'missing'."""
    if value is None:
        return False
    if isinstance(value, float) and pd.isna(value):
        return False
    if isinstance(value, (str, list, dict)) and len(value) == 0:
        return False
    return True


def completeness_report(df, field_map, record_type):
    rows = []
    n = len(df)
    for label, col in field_map.items():
        if col not in df.columns:
            rows.append({'field': label, 'column': col, 'present_pct': None,
                         'note': 'COLUMN NOT FOUND -- check FIELD_MAP'})
            continue
        non_empty = df[col].apply(is_non_empty).sum()
        rows.append({
            'field': label,
            'column': col,
            'present_pct': round(100 * non_empty / n, 1) if n else None,
            'note': '',
        })

    report = pd.DataFrame(rows)
    report.insert(0, 'record_type', record_type)
    return report


def add_any_nuts_row(df, report, record_type):
    """Jobs only: a record counts as having 'location with NUTS' if AT
    LEAST ONE of nuts1/nuts2/nuts3 is non-empty. Reported separately from
    the three individual nuts1/2/3 completeness rows above."""
    nuts_cols = [c for c in ['nuts1', 'nuts2', 'nuts3'] if c in df.columns]
    if not nuts_cols:
        return report
    n = len(df)
    any_nuts = df[nuts_cols].apply(lambda row: any(is_non_empty(v) for v in row), axis=1).sum()
    extra = pd.DataFrame([{
        'record_type': record_type, 'field': 'location_with_any_nuts',
        'column': '/'.join(nuts_cols), 'present_pct': round(100 * any_nuts / n, 1) if n else None,
        'note': '',
    }])
    return pd.concat([report, extra], ignore_index=True)


profile_report = completeness_report(profiles, PROFILE_FIELDS, 'profiles')
posting_report = completeness_report(postings, POSTING_FIELDS, 'postings')
posting_report = add_any_nuts_row(postings, posting_report, 'postings')

overall = pd.concat([profile_report, posting_report], ignore_index=True)
overall

,record_type,field,column,present_pct,note
0,profiles,skills,skills,62.7,
1,profiles,occupation_uris,occupation_uris,3.3,
2,profiles,content,content,95.9,
3,profiles,description,description,0.2,
4,profiles,location,location,100.0,
5,profiles,date,startdate,1.5,
6,profiles,source,source,100.0,
7,postings,skills,skills,80.4,
8,postings,occupations,occupations,83.0,
9,postings,description,description,92.8,


## Per-Source Breakdown (Week 2)

The aggregate table above can hide a lot: a field might look reasonably complete overall while being near-empty for one specific source and near-perfect for another. This section breaks completeness down by `source`, and checks two things the brief specifically asks for:

1. Does Revelio show a disproportionately **lower** `skills` completeness than other profile sources?
2. Does posting coverage (`upload_date`) genuinely start from **2019 onward**, or is that assumption wrong for some sources?

In [4]:
def completeness_by_source(df, field_map, source_col='source'):
    """
    Returns a tidy DataFrame: one row per (source, field), with present_pct
    and the raw n for that source, so small sources aren't over-interpreted.
    """
    rows = []
    for source_value, group in df.groupby(source_col):
        n = len(group)
        for label, col in field_map.items():
            if col not in group.columns:
                continue
            non_empty = group[col].apply(is_non_empty).sum()
            rows.append({
                'source': source_value,
                'field': label,
                'n': n,
                'present_pct': round(100 * non_empty / n, 1) if n else None,
            })
    return pd.DataFrame(rows)


profile_by_source = completeness_by_source(profiles, PROFILE_FIELDS)
posting_by_source = completeness_by_source(postings, POSTING_FIELDS)

# Pivot to source x field matrices -- easier to scan than a long table
profile_pivot = profile_by_source.pivot(index='source', columns='field', values='present_pct')
posting_pivot = posting_by_source.pivot(index='source', columns='field', values='present_pct')

# Attach sample size per source for context (a 99% on n=1 means nothing)
profile_n = profiles.groupby('source').size().rename('n')
posting_n = postings.groupby('source').size().rename('n')

profile_pivot = profile_n.to_frame().join(profile_pivot)
posting_pivot = posting_n.to_frame().join(posting_pivot)

print('=== Profiles: completeness by source ===')
display(profile_pivot.sort_values('n', ascending=False))

print('\n=== Postings: completeness by source ===')
display(posting_pivot.sort_values('n', ascending=False))

=== Profiles: completeness by source ===


,n,content,date,description,location,occupation_uris,skills,source
source,,,,,,,,
STACK-Biology,250,100.0,0.0,0.0,100.0,0.0,84.0,100.0
STACK-Chemistry,250,100.0,0.0,0.0,100.0,0.0,60.8,100.0
STACK-Earth Science,250,100.0,0.0,0.0,100.0,0.0,35.6,100.0
STACK-Electrical Engineering,250,100.0,0.0,0.0,100.0,0.0,88.4,100.0
STACK-Interpersonal Skills,250,100.0,0.0,0.0,100.0,0.0,32.0,100.0
STACK-Law,250,100.0,0.0,0.0,100.0,0.0,54.4,100.0
STACK-Linguistics,250,100.0,0.0,0.0,100.0,0.0,45.6,100.0
STACK-Literature,250,100.0,0.0,0.0,100.0,0.0,20.8,100.0
STACK-Mathematics,250,100.0,0.0,0.0,100.0,0.0,96.8,100.0



=== Postings: completeness by source ===


,n,date,description,location,nuts1,nuts2,nuts3,occupations,skills,source
source,,,,,,,,,,
OJA,468,100.0,0.0,100.0,0.0,0.0,0.0,100.0,100.0,100.0
eures,468,100.0,98.7,98.1,96.2,95.3,95.1,60.7,37.8,100.0
eures-escox,468,100.0,97.2,92.5,92.9,89.1,87.2,82.3,100.0,100.0
jobbland.se,468,100.0,100.0,100.0,0.0,0.0,0.0,80.6,79.9,100.0
jobbland,468,100.0,100.0,100.0,0.0,0.0,0.0,80.6,79.9,100.0
jobs.de,468,100.0,100.0,100.0,0.0,0.0,0.0,97.9,94.0,100.0
jobmedic.co.uk,468,100.0,100.0,100.0,0.0,0.0,0.0,93.8,80.8,100.0
jobmedic,468,100.0,100.0,100.0,0.0,0.0,0.0,93.8,80.8,100.0
kariera.fr,468,100.0,100.0,100.0,0.0,0.0,0.0,96.4,93.6,100.0


### Check 1: Is Revelio's `skills` completeness disproportionately low?

In [5]:
revelio_skills_pct = profile_pivot.loc['revelio', 'skills'] if 'revelio' in profile_pivot.index else None
other_sources = profile_pivot.drop(index='revelio', errors='ignore')
other_mean = other_sources['skills'].mean()
other_median = other_sources['skills'].median()

print(f"Revelio skills completeness: {revelio_skills_pct}%")
print(f"All other profile sources -- mean: {other_mean:.1f}%, median: {other_median:.1f}%")

if revelio_skills_pct is not None:
    gap = other_mean - revelio_skills_pct
    print(f"\nGap: Revelio is {gap:.1f} percentage points "
          f"{'LOWER' if gap > 0 else 'HIGHER'} than the average of other sources.")

Revelio skills completeness: 12.4%
All other profile sources -- mean: 64.1%, median: 62.8%

Gap: Revelio is 51.7 percentage points LOWER than the average of other sources.


### Check 2: Does posting coverage genuinely start from 2019 onward?

In [6]:
postings['upload_date_parsed'] = pd.to_datetime(postings['upload_date'], errors='coerce')

date_range_by_source = postings.groupby('source')['upload_date_parsed'].agg(['min', 'max', 'count'])
date_range_by_source = date_range_by_source.sort_values('min')

print('=== Posting date range by source ===')
display(date_range_by_source)

overall_min = postings['upload_date_parsed'].min()
overall_max = postings['upload_date_parsed'].max()
print(f"\nOverall posting date range in this sample: {overall_min.date()} to {overall_max.date()}")

cutoff = pd.Timestamp('2019-01-01')
sources_not_reaching_2019 = date_range_by_source[date_range_by_source['min'] > cutoff]
if len(sources_not_reaching_2019):
    print(f"\n{len(sources_not_reaching_2019)} source(s) do NOT have data reaching back to 2019:")
    display(sources_not_reaching_2019)
else:
    print("\nAll sources have at least some data at or before 2019-01-01.")

=== Posting date range by source ===


,min,max,count
source,,,
OJA,2019-05-01,2019-05-05,468
eures-escox,2022-02-08,2026-02-17,468
eures,2023-01-18,2025-12-30,468
jobbland,2023-05-31,2024-10-28,468
jobbland.se,2023-05-31,2024-10-28,468
brightminds,2024-08-11,2024-08-11,1
jobbguru,2024-08-11,2024-09-12,319
jobbguru.se,2024-08-11,2024-09-12,319
jobmedic,2024-08-11,2024-09-21,468



Overall posting date range in this sample: 2019-05-01 to 2026-04-01

16 source(s) do NOT have data reaching back to 2019:


,min,max,count
source,,,
OJA,2019-05-01,2019-05-05,468
eures-escox,2022-02-08,2026-02-17,468
eures,2023-01-18,2025-12-30,468
jobbland,2023-05-31,2024-10-28,468
jobbland.se,2023-05-31,2024-10-28,468
brightminds,2024-08-11,2024-08-11,1
jobbguru,2024-08-11,2024-09-12,319
jobbguru.se,2024-08-11,2024-09-12,319
jobmedic,2024-08-11,2024-09-21,468


## Next step (Week 2)

Once field names are confirmed and this runs cleanly, the brief's Step 2 asks for this same audit **disaggregated by source, country, and sector**, plus an explicit check that Revelio-sourced profiles show a higher empty-`skills` rate than other sources, and a check that posting coverage starts from 2019 onward. That's a natural next notebook (`03_disaggregated_audit.ipynb`) once this one is confirmed working against real data.